Control N°1 Optimización


Importacion de librerias y seleccion del Solver

In [1]:
%pip install -q amplpy
from amplpy import AMPL, ampl_notebook



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
ampl = ampl_notebook(modules=['highs'], license_uuid = "default")

AMPL Version 20250901 (MSVC 19.44.35215.0, 64-bit)
Demo license with maintenance expiring 20270131.
Using license file "C:\Users\Manguera\AppData\Roaming\Python\Python313\site-packages\ampl_module_base\bin\ampl.lic".



## **Problema N°1: Optimización de Procesos de Producción**

El objetivo es determinar las horas de operación de dos procesos distintos para minimizar el costo total diario, asegurando que se cumplan las demandas mínimas de tres químicos específicos.

### **Variables de Decisión**
* $x_1$: Horas de operación del **Proceso 1** por día.
* $x_2$: Horas de operación del **Proceso 2** por día.

### **Función Objetivo**
Minimizar el costo total operativo diario ($Z$):
$$\text{Min } Z = 40x_1 + 10x_2$$

### **Restricciones**
Sujeto a:

1. **R1** (Demanda del químico A): $3x_1 + x_2 \geq 40$
2. **R2** (Demanda del químico B): $x_1 + x_2 \geq 15$
3. **R3** (Demanda del químico C): $x_1 \geq 5$
4. **R4** (No negatividad): $x_1, x_2 \geq 0$

In [ ]:
# Definir el modelo AMPL
model ="""
var x1 >= 0;  # Horas del proceso 1
var x2 >= 0;  # Horas del proceso 2

# Función objetivo:
minimize costo: 40*x1 + 10*x2;

# Restricciones
s.t.
  demanda_A: 3*x1 + x2 >= 40;    # Demanda del químico A
  demanda_B: x1 + x2 >= 15;      # Demanda del químico B
  demanda_C: x1 >= 5;             # Demanda del químico C
"""

# Resolver el modelo

ampl.reset()
ampl.eval(model)
ampl.option["solver"] = "highs"
ampl.solve()

# Mostrar resultados
print("Estado solución:", ampl.solve_result)
print(f"\nValor óptimo: {ampl.obj['costo'].value():.2f}")
print(f"\nVariables de decisión:")
print(f"x1 (Proceso 1): {ampl.var['x1'].value():.2f} horas")
print(f"x2 (Proceso 2): {ampl.var['x2'].value():.2f} horas")

HiGHS 1.11.0HiGHS 1.11.0: optimal solution; objective 450
0 simplex iterations
0 barrier iterations
Estado solución: solved

Valor óptimo (Z): 450.00

Variables de decisión:
x1 (Proceso 1): 5.00 horas
x2 (Proceso 2): 25.00 horas


## Problema 2: Mezcla y Procesamiento con Subproductos

Este es un problema de Programación Lineal (PL) de mezcla y procesamiento con subproductos (desechos) que deben ser balanceados o gestionados.

### Variables de Decisión
* $M_1$: Libras de materia prima compradas para elaborar el Insumo 1.
* $M_2$: Libras de materia prima compradas para elaborar el Insumo 2.
* $A$: Onzas de Producto A producidas (procesos tipo 1).
* $B$: Onzas de Producto B producidas (procesos tipo 2).
* $C$: Onzas de Producto C fabricadas.
* $D$: Onzas de Producto D fabricadas.
* $R$: Onzas de desecho líquido vertidas al río.

### Función Objetivo
Maximizar la utilidad ($Z$), que es el ingreso por ventas menos los costos totales (costo de compra y procesamiento de materia prima, más costos de los procesos de los productos):
$$\text{Max } Z = 17A + 16B + 7C + 2D - 8M_1 - 10M_2$$

### Restricciones
Sujeto a:
$$ 
\begin{align*}
2M_1 &= 2A + B + 2C && \text{(Balance de Insumo 1)} \\
3M_2 &= A + 2B + 2D && \text{(Balance de Insumo 2)} \\
A + 0.8B &= R + 0.8C + 1.2D && \text{(Balance de Desecho Líquido)} \\
R &\leq 1000 && \text{(Límite Gubernamental del Río)} \\
A &\leq 5000 && \text{(Demanda Máxima Producto A)} \\
B &\leq 5000 && \text{(Demanda Máxima Producto B)} \\
2M_1 + 2M_2 + 2A + 3B + C + D &\leq 6000 && \text{(Disponibilidad de Tiempo Total)} \\
M_1, M_2, A, B, C, D, R &\geq 0 && \text{(No negatividad)}
\end{align*}
$$

In [ ]:
# Definir el modelo AMPL para el segundo problema
model2 = """
var M1 >= 0;  # Libras de materia prima para Insumo 1
var M2 >= 0;  # Libras de materia prima para Insumo 2
var A >= 0;   # Onzas de Producto A
var B >= 0;   # Onzas de Producto B
var C >= 0;   # Onzas de Producto C
var D >= 0;   # Onzas de Producto D
var R >= 0;   # Onzas de desecho líquido

# Función objetivo: Maximizar utilidad
maximize utilidad: 17*A + 16*B + 7*C + 2*D - 8*M1 - 10*M2;

# Restricciones
s.t.
  balance_insumo1: 2*M1 = 2*A + B + 2*C;          # Balance de Insumo 1
  balance_insumo2: 3*M2 = A + 2*B + 2*D;          # Balance de Insumo 2
  balance_desecho: A + 0.8*B = R + 0.8*C + 1.2*D; # Balance de Desecho Líquido
  limite_rio: R <= 1000;                           # Límite Gubernamental del Río
  demanda_A: A <= 5000;                            # Demanda Máxima Producto A
  demanda_B: B <= 5000;                            # Demanda Máxima Producto B
  tiempo_total: 2*M1 + 2*M2 + 2*A + 3*B + C + D <= 6000;  # Disponibilidad de Tiempo Total
"""

# Resolver el modelo
ampl.reset()
ampl.eval(model2)
ampl.option["solver"] = "highs"
ampl.solve()

# Mostrar resultados
print("Estado solución:", ampl.solve_result)
print(f"\nValor óptimo de la utilidad: {ampl.obj['utilidad'].value():.2f}")
print(f"\nVariables de decisión:")
print(f"M1 (Materia prima Insumo 1): {ampl.var['M1'].value():.2f} libras")
print(f"M2 (Materia prima Insumo 2): {ampl.var['M2'].value():.2f} libras")
print(f"A (Producto A): {ampl.var['A'].value():.2f} onzas")
print(f"B (Producto B): {ampl.var['B'].value():.2f} onzas")
print(f"C (Producto C): {ampl.var['C'].value():.2f} onzas")
print(f"D (Producto D): {ampl.var['D'].value():.2f} onzas")
print(f"R (Desecho líquido): {ampl.var['R'].value():.2f} onzas")

HiGHS 1.11.0HiGHS 1.11.0: optimal solution; objective 6366.336634
2 simplex iterations
0 barrier iterations
Estado solución: solved

Valor óptimo de la utilidad: 6366.34

Variables de decisión:
M1 (Materia prima Insumo 1): 1356.44 libras
M2 (Materia prima Insumo 2): 386.14 libras
A (Producto A): 1158.42 onzas
B (Producto B): 0.00 onzas
C (Producto C): 198.02 onzas
D (Producto D): 0.00 onzas
R (Desecho líquido): 1000.00 onzas


## Problema 3: Refinería Melrose

Este es un modelo complejo de mezcla (blending) donde los subproductos obtenidos pueden ser transformados mediante un proceso de mejora (desintegración catalítica).

### Variables de Decisión
* $BC_1, BC_2, BC_3$: Barriles de **C**rudo procesados mediante los métodos 1, 2 y 3.
* $M_{6 \to 8}, M_{8 \to 10}$: Barriles de crudo sometidos a **M**ejora para subir de grado.
* $Gas_6, Gas_8, Gas_{10}$: Barriles de grado 6, 8 y 10 destinados a la mezcla de **Gas**olina final.
* $Ace_6, Ace_8, Ace_{10}$: Barriles de grado 6, 8 y 10 destinados a la mezcla de **Ace**ite final.
* $BD_6, BD_8, BD_{10}$: Barriles de grado 6, 8 y 10 **D**esechados.

### Función Objetivo
Maximizar la utilidad ($Z$), calculada como los ingresos por venta de gasolina (\$8/barril) y aceite (\$6/barril), menos los costos operativos (procesamiento, mejora y desecho).

*(Nota: Como no se entregaron los coeficientes exactos de costos en el enunciado, se expresan de manera conceptual en la función).*
$$\text{Max } Z = 8(Gas_6 + Gas_8 + Gas_{10}) + 6(Ace_6 + Ace_8 + Ace_{10}) - \text{Costos de Procesos, Mejora y Desecho}$$

### Restricciones
Sujeto a:
$$ 
\begin{align*}
\text{Salida Grado } i &= Gas_i + Ace_i + M_{i \to j} + BD_i && \text{(Equilibrio de salidas hacia Venta, Mejora y Desecho)} \\
-3Gas_6 - Gas_8 + Gas_{10} &\geq 0 && \text{(Grado Gasolina } \geq 9 \text{, linealizada)} \\
-Ace_6 + Ace_8 + 3Ace_{10} &\geq 0 && \text{(Grado Aceite } \geq 7 \text{, linealizada)} \\
BC_i, M_{i \to j}, Gas_i, Ace_i, BD_i &\geq 0 && \text{(No negatividad)}
\end{align*}
$$

El cálculo original del grado promedio ponderado para la gasolina es una fracción: 
$\frac{6Gas_6 + 8Gas_8 + 10Gas_{10}}{Gas_6 + Gas_8 + Gas_{10}} \geq 9$


In [ ]:
model_melrose = """
# Variables de Barriles de Crudo
var BC1 >= 0; var BC2 >= 0; var BC3 >= 0;

# Variables de Mejora
var M6_8 >= 0; var M8_10 >= 0;

# Variables de Gasolina y Aceite
var Gas6 >= 0; var Gas8 >= 0; var Gas10 >= 0;
var Ace6 >= 0; var Ace8 >= 0; var Ace10 >= 0;

# Variables de Barriles de Desecho
var BD6 >= 0; var BD8 >= 0; var BD10 >= 0;

maximize utilidad: 
    8*(Gas6 + Gas8 + Gas10) + 6*(Ace6 + Ace8 + Ace10) 
    - (3.4*BC1 + 3.0*BC2 + 2.6*BC3)
    - 1.3*M6_8 - 2.0*M8_10 
    - 0.2*(BD6 + BD8 + BD10);

s.t.
# Equilibrio de salidas hacia Venta, Mejora y Desecho
bal_6:  0.2*BC1 + 0.3*BC2 + 0.4*BC3 = Gas6 + Ace6 + M6_8 + BD6;
bal_8:  0.2*BC1 + 0.3*BC2 + 0.4*BC3 + M6_8 = Gas8 + Ace8 + M8_10 + BD8;
bal_10: 0.6*BC1 + 0.4*BC2 + 0.2*BC3 + M8_10 = Gas10 + Ace10 + BD10;

# Restricciones de demanda máxima
max_gasolina: Gas6 + Gas8 + Gas10 <= 2000;
max_aceite: Ace6 + Ace8 + Ace10 <= 600;

# Calidad: el peso ponderado debe ser mayor o igual al target (9 para gas, 7 para aceite)
calidad_gasolina: -3*Gas6 - 1*Gas8 + 1*Gas10 >= 0; 
calidad_aceite:   -1*Ace6 + 1*Ace8 + 3*Ace10 >= 0;
"""

# Ejecución de AMPL
ampl.reset()
ampl.eval(model_melrose)
ampl.option["solver"] = "highs"
ampl.solve()

# Extracción de resultados usando las nuevas variables
gasolina = sum([ampl.get_value(f'Gas{i}') for i in [6,8,10]])
aceite = sum([ampl.get_value(f'Ace{i}') for i in [6,8,10]])

print(f"Barriles Gasolina Final: {gasolina}")
print(f"Barriles Aceite Final: {aceite}")
print(f"Utilidad Maximizada: USD {ampl.get_value('utilidad')}")

HiGHS 1.11.0HiGHS 1.11.0: optimal solution; objective 11230
9 simplex iterations
0 barrier iterations
Barriles Gasolina Final: 2000
Barriles Aceite Final: 600
Utilidad Maximizada: USD 11230


## Problema N°4: Donovan Enterprises fabrica de licuadoras


Este modelo busca optimizar la contratación de empleados y la gestión de inventario para cumplir con la demanda trimestral de licuadoras al menor costo posible.

### Variables de Decisión

**Fuerza Laboral (Patrones de Contratación)**
* $E_1$: Empleados contratados que tienen libre el Trimestre 1 (trabajan Q2, Q3, Q4).
* $E_2$: Empleados contratados que tienen libre el Trimestre 2 (trabajan Q1, Q3, Q4).
* $E_3$: Empleados contratados que tienen libre el Trimestre 3 (trabajan Q1, Q2, Q4).
* $E_4$: Empleados contratados que tienen libre el Trimestre 4 (trabajan Q1, Q2, Q3).

**Producción e Inventario**
* $P_1, P_2, P_3, P_4$: Licuadoras producidas durante los trimestres 1, 2, 3 y 4.
* $I_1, I_2, I_3, I_4$: Licuadoras en inventario al final de los trimestres 1, 2, 3 y 4.

### Función Objetivo
Minimizar el costo total ($Z$), compuesto por los salarios anuales (\$30,000 por empleado) y los costos de mantener inventario al final de cada trimestre (\$30 por unidad).

$$\text{Min } Z = 30000(E_1 + E_2 + E_3 + E_4) + 30(I_1 + I_2 + I_3 + I_4)$$

### Restricciones
Sujeto a:

$$ 
\begin{align*}
P_1 &\leq 500(E_2 + E_3 + E_4) && \text{(Capacidad Trimestre 1)} \\
P_2 &\leq 500(E_1 + E_3 + E_4) && \text{(Capacidad Trimestre 2)} \\
P_3 &\leq 500(E_1 + E_2 + E_4) && \text{(Capacidad Trimestre 3)} \\
P_4 &\leq 500(E_1 + E_2 + E_3) && \text{(Capacidad Trimestre 4)} \\
I_1 &= 600 + P_1 - 4000 && \text{(Balance Trimestre 1 con Inv. Inicial de 600)} \\
I_2 &= I_1 + P_2 - 2000 && \text{(Balance Trimestre 2)} \\
I_3 &= I_2 + P_3 - 3000 && \text{(Balance Trimestre 3)} \\
I_4 &= I_3 + P_4 - 10000 && \text{(Balance Trimestre 4)} \\
E_i, P_i, I_i &\geq 0 && \text{(No negatividad)}
\end{align*}
$$

In [14]:
modelo_donovan = """

# VARIABLES (Declaradas como enteras para mayor realismo)

var E1 integer >= 0; 
var E2 integer >= 0; 
var E3 integer >= 0; 
var E4 integer >= 0;

var P1 integer >= 0; 
var P2 integer >= 0; 
var P3 integer >= 0; 
var P4 integer >= 0;

var I1 integer >= 0; 
var I2 integer >= 0; 
var I3 integer >= 0; 
var I4 integer >= 0;


# FUNCIÓN OBJETIVO

minimize Costo_Total: 
    30000 * (E1 + E2 + E3 + E4) + 30 * (I1 + I2 + I3 + I4);


# RESTRICCIONES

s.t. 
cap_T1: P1 <= 500 * (E2 + E3 + E4);
cap_T2: P2 <= 500 * (E1 + E3 + E4);
cap_T3: P3 <= 500 * (E1 + E2 + E4);
cap_T4: P4 <= 500 * (E1 + E2 + E3);

bal_T1: I1 = 600 + P1 - 4000;
bal_T2: I2 = I1 + P2 - 2000;
bal_T3: I3 = I2 + P3 - 3000;
bal_T4: I4 = I3 + P4 - 10000;
"""

# Ejecución de AMPL
ampl.reset()
ampl.eval(modelo_donovan)
ampl.option["solver"] = "highs"  
ampl.solve()


# Mostrar resultados
print("\n--- PLAN DE CONTRATACIÓN ---")
print(f"Empleados libres en Q1 (E1): {ampl.get_value('E1'):.0f}")
print(f"Empleados libres en Q2 (E2): {ampl.get_value('E2'):.0f}")
print(f"Empleados libres en Q3 (E3): {ampl.get_value('E3'):.0f}")
print(f"Empleados libres en Q4 (E4): {ampl.get_value('E4'):.0f}")
total_empleados = sum([ampl.get_value(f'E{i}') for i in range(1, 5)])
print(f"Total Empleados Contratados: {total_empleados:.0f}")

print("\n--- PLAN DE PRODUCCIÓN E INVENTARIO ---")
for i in range(1, 5):
    prod = ampl.get_value(f'P{i}')
    inv = ampl.get_value(f'I{i}')
    print(f"Trimestre {i}: Producción = {prod:.0f} | Inventario Final = {inv:.0f}")

print(f"\n>>> COSTO TOTAL OPTIMIZADO: USD ${ampl.get_value('Costo_Total'):,.2f} <<<")

HiGHS 1.11.0HiGHS 1.11.0: optimal solution; objective 495000
6 simplex iterations
1 branching nodes

--- PLAN DE CONTRATACIÓN ---
Empleados libres en Q1 (E1): 4
Empleados libres en Q2 (E2): 9
Empleados libres en Q3 (E3): 0
Empleados libres en Q4 (E4): 0
Total Empleados Contratados: 13

--- PLAN DE PRODUCCIÓN E INVENTARIO ---
Trimestre 1: Producción = 3400 | Inventario Final = 0
Trimestre 2: Producción = 2000 | Inventario Final = 0
Trimestre 3: Producción = 6500 | Inventario Final = 3500
Trimestre 4: Producción = 6500 | Inventario Final = 0

>>> COSTO TOTAL OPTIMIZADO: USD $495,000.00 <<<


## Problema 5: Optimización de Encuestas Telefónicas

El objetivo de este modelo es determinar la cantidad óptima de llamadas diurnas y nocturnas a realizar para cumplir con las cuotas demográficas de una investigación de mercado, minimizando el costo total de la operación.

### Variables de Decisión
* $D$: Número de llamadas realizadas durante el **D**ía.
* $N$: Número de llamadas realizadas durante la **N**oche.

### Función Objetivo
Minimizar el costo total ($Z$), dado que cada llamada de día cuesta \$2 y cada llamada de noche cuesta \$5:
$$\text{Min } Z = 2D + 5N$$

### Restricciones
Las probabilidades de la tabla representan la tasa de éxito de contactar a cada perfil por cada llamada realizada. Por ejemplo, el 30% (0.30) de las llamadas diurnas son contestadas por esposas.

Sujeto a:
$$ 
\begin{align*}
0.30D + 0.30N &\geq 150 && \text{(Meta mínima de Esposas)} \\
0.10D + 0.30N &\geq 120 && \text{(Meta mínima de Esposos)} \\
0.10D + 0.15N &\geq 100 && \text{(Meta mínima de Varones Solteros)} \\
0.10D + 0.20N &\geq 110 && \text{(Meta mínima de Mujeres Solteras)} \\
N &\leq D && \text{(Límite de horario: máximo 50\% de las llamadas pueden ser nocturnas)*} \\
D, N &\geq 0 && \text{(No negatividad)}
\end{align*}
$$



In [15]:
modelo_encuesta = """

# VARIABLES (Declaradas como enteras para representar llamadas completas)

var D integer >= 0; # Llamadas de Día
var N integer >= 0; # Llamadas de Noche


# FUNCIÓN OBJETIVO

# Minimizar el costo: $2 de día y $5 de noche
minimize Costo_Total: 
    2 * D + 5 * N;




# 1. Metas demográficas (Tasa de éxito * Total de llamadas >= Meta)
s.t. 
meta_esposas:         0.30 * D + 0.30 * N >= 150;
meta_esposos:         0.10 * D + 0.30 * N >= 120;
meta_varon_soltero:   0.10 * D + 0.15 * N >= 100;
meta_mujer_soltera:   0.10 * D + 0.20 * N >= 110;

# 2. Límite de personal (Las llamadas nocturnas no pueden superar la mitad del total)
# Expresado de forma lineal y simplificada: N <= D
s.t. 
maximo_nocturnas: N <= D;
"""

# Ejecución de AMPL
ampl.reset()
ampl.eval(modelo_encuesta)
ampl.option["solver"] = "highs"  
ampl.solve()


llamadas_dia = ampl.get_value('D')
llamadas_noche = ampl.get_value('N')
costo_optimo = ampl.get_value('Costo_Total')

# Mostrar resultados
print("\n--- PLAN DE LLAMADAS ---")
print(f"Llamadas durante el Día (D): {llamadas_dia:.0f}")
print(f"Llamadas durante la Noche (N): {llamadas_noche:.0f}")
print(f"Total de llamadas proyectadas: {(llamadas_dia + llamadas_noche):.0f}")

print(f"\n>>> COSTO TOTAL MINIMIZADO: USD ${costo_optimo:,.2f} <<<")

HiGHS 1.11.0HiGHS 1.11.0: optimal solution; objective 2300
3 simplex iterations
1 branching nodes

--- PLAN DE LLAMADAS ---
Llamadas durante el Día (D): 900
Llamadas durante la Noche (N): 100
Total de llamadas proyectadas: 1000

>>> COSTO TOTAL MINIMIZADO: USD $2,300.00 <<<
